# OSM Power Plants: Data Quality Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/open-energy-transition/osm-powerplants/blob/main/notebooks/workshop_data_quality.ipynb)
[![PyPI](https://img.shields.io/pypi/v/osm-powerplants)](https://pypi.org/project/osm-powerplants/)

This workshop demonstrates how to:
1. **Extract** power plant data from OpenStreetMap
2. **Analyze** why certain plants fail validation
3. **Export** issues for fixing in JOSM
4. **Visualize** data on interactive maps
5. **Verify** improvements after OSM edits

---

## Why This Matters

The `osm-powerplants` package feeds into [powerplantmatching](https://github.com/PyPSA/powerplantmatching) → [PyPSA-Eur](https://github.com/PyPSA/pypsa-eur), enabling open energy system modeling. Better OSM data = better energy models.

**Key innovation**: Rejection tracking tells you *why* data fails, not just *that* it fails.

## 1. Setup

In [ ]:
# Install osm-powerplants from PyPI
!pip install -q osm-powerplants folium

print("Installation complete!")

In [ ]:
# Imports
import folium
import pandas as pd
from IPython.display import display, HTML

from osm_powerplants import process_units, get_config, get_cache_dir, Units
from osm_powerplants.models import RejectionReason
from osm_powerplants.quality.rejection import RejectionTracker
from osm_powerplants.retrieval.client import OverpassAPIClient
from osm_powerplants.workflow import Workflow

print("Imports ready!")

## 2. Extract Power Plant Data

We'll use **Malta** for this demo — it's small and processes quickly (~30 seconds).

In [ ]:
# Configuration
COUNTRY = "Malta"  # Try: Luxembourg, Cyprus, Estonia

# Quick extraction with default settings
config = get_config()
cache_dir = str(get_cache_dir(config))

print(f"Fetching power plants for {COUNTRY}...")
df = process_units(
    countries=[COUNTRY],
    config=config,
    cache_dir=cache_dir,
)

print(f"\nFound {len(df)} valid power plants")
print(f"Total capacity: {df['Capacity'].sum():.1f} MW")

In [ ]:
# View the data
display(df.head(10))

In [ ]:
# Breakdown by fuel type
fuel_summary = df.groupby('Fueltype').agg(
    Count=('projectID', 'count'),
    Capacity_MW=('Capacity', 'sum')
).sort_values('Capacity_MW', ascending=False)

display(fuel_summary)

## 3. Rejection Analysis

This is the **key feature** of osm-powerplants. We re-run with strict settings to capture all rejections and understand data quality issues.

In [ ]:
# Strict settings to capture all rejections
strict_config = get_config()
strict_config["missing_name_allowed"] = False
strict_config["missing_technology_allowed"] = False
strict_config["missing_start_date_allowed"] = False

# Initialize tracking
tracker = RejectionTracker()
units = Units()

print(f"Analyzing rejections for {COUNTRY}...")

with OverpassAPIClient(cache_dir=cache_dir) as client:
    workflow = Workflow(
        client=client,
        rejection_tracker=tracker,
        units=units,
        config=strict_config,
    )
    workflow.process_country_data(COUNTRY)

print(f"\nResults:")
print(f"  Valid units: {len(units)}")
print(f"  Rejected elements: {tracker.get_total_count()}")

In [ ]:
# Detailed rejection breakdown
print("Rejection Summary:\n")
print(tracker.get_summary_string())

### Understanding Rejection Reasons

| Rejection Reason | What It Means | OSM Fix |
|-----------------|---------------|----------|
| `Missing source type` | No fuel type specified | Add `plant:source=solar/wind/gas/...` |
| `Missing technology type` | No technology specified | Add `plant:method=photovoltaic/run_of_river/...` |
| `Missing output tag` | No capacity specified | Add `plant:output:electricity=X MW` |
| `Capacity placeholder value` | Capacity is 0 or placeholder | Update with actual MW value |
| `Missing name tag` | No name tag | Add `name=Plant Name` |
| `Element within existing plant geometry` | Generator inside plant boundary | Not an error — already counted |

## 4. Export for JOSM

Export rejected elements as GeoJSON files that can be loaded into JOSM as a hint layer.

In [ ]:
import os

# Export all rejections to a single file
output_file = f"{COUNTRY.lower()}_rejections.geojson"
tracker.save_geojson(output_file)
print(f"Saved: {output_file}")

# Export by reason (for targeted fixing)
os.makedirs("by_reason", exist_ok=True)
tracker.save_geojson_by_reasons("by_reason", prefix=COUNTRY.lower())
print(f"Saved per-reason files in: by_reason/")

# List generated files
for f in sorted(os.listdir("by_reason")):
    print(f"  {f}")

In [ ]:
# Download files (Colab)
try:
    from google.colab import files
    print("Downloading GeoJSON files...")
    files.download(output_file)
except ImportError:
    print(f"Not running in Colab. File saved to: {output_file}")

## 5. Interactive Map - Rejections

Visualize rejected elements on an interactive map with direct links to edit in OSM.

In [ ]:
# Get GeoJSON data
geojson = tracker.generate_geojson()

if geojson['features']:
    # Calculate map center from features
    lats = [f['geometry']['coordinates'][1] for f in geojson['features']]
    lons = [f['geometry']['coordinates'][0] for f in geojson['features']]
    center_lat = sum(lats) / len(lats)
    center_lon = sum(lons) / len(lons)
    
    # Create map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
    
    # Color by rejection reason
    colors = {
        'Missing source type': 'red',
        'Missing technology type': 'orange',
        'Missing output tag': 'purple',
        'Capacity placeholder value': 'blue',
        'Missing name tag': 'gray',
        'Element within existing plant geometry': 'green',
    }
    
    for feature in geojson['features']:
        coords = feature['geometry']['coordinates']
        props = feature['properties']
        reason = props.get('rejection_reason', 'Unknown')
        osm_url = props.get('osm_element', '#')
        
        popup = f"""
        <b>Reason:</b> {reason}<br>
        <b>Keywords:</b> {props.get('rejection_keywords', 'N/A')}<br>
        <a href="{osm_url}" target="_blank">Edit in OSM</a>
        """
        
        folium.CircleMarker(
            location=[coords[1], coords[0]],
            radius=8,
            color=colors.get(reason, 'gray'),
            fill=True,
            fillOpacity=0.7,
            popup=folium.Popup(popup, max_width=300)
        ).add_to(m)
    
    print(f"Showing {len(geojson['features'])} rejected elements:")
    display(m)
else:
    print("No rejections to display — perfect data quality!")

## 6. Interactive Map - Valid Plants

Visualize the valid power plants by fuel type.

In [ ]:
if not df.empty:
    center_lat = df['lat'].mean()
    center_lon = df['lon'].mean()
    
    m2 = folium.Map(location=[center_lat, center_lon], zoom_start=10)
    
    fuel_colors = {
        'Solar': 'orange',
        'Wind': 'blue',
        'Natural Gas': 'gray',
        'Hydro': 'cyan',
        'Oil': 'black',
        'Bioenergy': 'green',
        'Solid Biomass': 'darkgreen',
        'Biogas': 'lightgreen',
    }
    
    for _, row in df.iterrows():
        fuel = row.get('Fueltype', 'Other')
        capacity = row.get('Capacity', 0) or 0
        name = row.get('Name', 'Unknown')
        
        popup = f"""
        <b>{name}</b><br>
        Fuel: {fuel}<br>
        Capacity: {capacity:.1f} MW
        """
        
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=max(5, min(20, capacity / 10)),
            color=fuel_colors.get(fuel, 'purple'),
            fill=True,
            fillOpacity=0.6,
            popup=folium.Popup(popup, max_width=300)
        ).add_to(m2)
    
    print(f"Valid power plants in {COUNTRY}:")
    display(m2)
else:
    print("No valid plants to display.")

## 7. JOSM Workflow

To fix the identified issues:

1. **Download** the GeoJSON file from the cell above
2. **Open JOSM** and load the area (e.g., Malta)
3. **File → Open** → select the GeoJSON as a hint layer
4. **Click** on markers to see what needs fixing
5. **Edit** the power plant and add missing tags
6. **Upload** your changes to OSM

### Common Fixes

```
# For solar plants:
plant:source = solar
plant:method = photovoltaic
plant:output:electricity = 5 MW

# For wind farms:
plant:source = wind
plant:method = horizontal_axis
plant:output:electricity = 12 MW
```

## 8. Verification (After Edits)

After making OSM edits, re-run with `force_refresh=True` to verify improvements.

Note: OSM data may take 5-10 minutes to propagate to Overpass API.

In [ ]:
# Uncomment to re-run after making OSM edits

# config_refresh = get_config()
# config_refresh["force_refresh"] = True
# 
# df_after = process_units(
#     countries=[COUNTRY],
#     config=config_refresh,
#     cache_dir=cache_dir,
# )
# 
# print(f"Before: {len(df)} plants, {df['Capacity'].sum():.1f} MW")
# print(f"After:  {len(df_after)} plants, {df_after['Capacity'].sum():.1f} MW")

print("Uncomment the code above after making OSM edits to verify improvements.")

## 9. Try Another Country

Change `COUNTRY` at the top and re-run all cells to analyze a different country.

In [ ]:
# Quick comparison of multiple small countries
test_countries = ["Malta", "Luxembourg", "Cyprus"]

results = []
for country in test_countries:
    try:
        df_test = process_units(
            countries=[country],
            config=config,
            cache_dir=cache_dir,
        )
        results.append({
            'Country': country,
            'Plants': len(df_test),
            'Capacity_MW': df_test['Capacity'].sum()
        })
    except Exception as e:
        print(f"Warning - {country}: {e}")

comparison = pd.DataFrame(results)
display(comparison)

---

## Resources

- **PyPI**: `pip install osm-powerplants`
- **Docs**: https://open-energy-transition.github.io/osm-powerplants
- **GitHub**: https://github.com/open-energy-transition/osm-powerplants
- **MapYourGrid**: https://mapyourgrid.org/strategies/#improve-osm-tags-with-ppm
- **powerplantmatching**: https://github.com/PyPSA/powerplantmatching

---

*Workshop created for the OSM Power Plants Final Project Report*